# 02 Evaluation

Purpose: Evaluate all five scoring variants on the test set, run alpha sensitivity for local DNDS, and save summary artifacts.

Inputs: `dataset/CVPR_2024_dataset_Test/`, `dataset_text/test.csv`, `models/mobilenetv3_best.pth`, `text_models/distilbert_best.pth`, populated `chroma_db/`.

Outputs: `results/phase2/evaluation_results.json`, `figures/phase2/scoring_comparison.png`, `figures/phase2/alpha_sensitivity.png`, `figures/phase2/confusion_matrices_phase2.png`, `figures/phase2/phase2_vs_phase1.png`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../..').resolve()))

import pandas as pd
import torch
from torchvision import models, transforms
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from src.phase2.db_client import get_image_collection, get_persistent_client, get_text_collection
from src.phase2.evaluation import evaluate_variant, save_results
from src.phase2.scoring import global_dnds, idw, local_dnds, majority_vote, traditional
from src.phase2.visualization import (
    plot_alpha_sensitivity,
    plot_confusion_matrices,
    plot_phase2_vs_phase1,
    plot_scoring_comparison,
)

CONFIG = {
    'k_vote': 10,
    'K_density': 50,
    'alpha': 0.5,
    'alpha_sweep': [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0],
    'imbalance_ratios': [10, 50, 100],
    'batch_size': 100,
    'epsilon': 1e-6,
    'db_path': './chroma_db',
    'figures_path': './figures/phase2',
    'class_names': ['Black', 'Blue', 'Green', 'TTR'],
    'minority_classes': ['TTR', 'Green'],
    'majority_classes': ['Black', 'Blue'],
}

REPO_ROOT = Path('../..').resolve()
TEST_DIR = REPO_ROOT / 'dataset' / 'CVPR_2024_dataset_Test'
TEST_CSV = REPO_ROOT / 'dataset_text' / 'test.csv'
RESULTS_PATH = REPO_ROOT / 'results' / 'phase2' / 'evaluation_results.json'
FIGURES_DIR = REPO_ROOT / 'figures' / 'phase2'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not TEST_DIR.exists() or not TEST_CSV.exists():
    raise FileNotFoundError('Test dataset missing. Expected dataset and dataset_text paths in repo root.')

In [ ]:
df = pd.read_csv(TEST_CSV)
test_samples = []
for row in df.itertuples(index=False):
    test_samples.append({
        'image_path': str(TEST_DIR / row.label / row.text),
        'text': row.text,
        'label': row.label,
    })

client = get_persistent_client(str(REPO_ROOT / 'chroma_db'))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)

variant_fns = {
    'majority_vote': majority_vote,
    'idw': idw,
    'global_dnds': global_dnds,
    'local_dnds': local_dnds,
}

In [ ]:
results = {'variants': {}, 'alpha_sweep': {'alphas': [], 'macro_f1': []}}

for name, fn in variant_fns.items():
    metrics = evaluate_variant(fn, test_samples, image_collection, text_collection, CONFIG)
    results['variants'][name] = metrics

# Alpha sweep for local DNDS
for alpha in CONFIG['alpha_sweep']:
    metrics = evaluate_variant(local_dnds, test_samples, image_collection, text_collection, CONFIG, alpha=alpha)
    results['alpha_sweep']['alphas'].append(alpha)
    results['alpha_sweep']['macro_f1'].append(metrics['macro_f1'])

In [ ]:
# Traditional baseline
image_ckpt = REPO_ROOT / 'models' / 'mobilenetv3_best.pth'
text_ckpt = REPO_ROOT / 'text_models' / 'distilbert_best.pth'

if image_ckpt.exists() and text_ckpt.exists():
    image_model = models.mobilenet_v3_large(weights=None)
    image_model.classifier[3] = torch.nn.Linear(image_model.classifier[3].in_features, len(CONFIG['class_names']))
    image_payload = torch.load(image_ckpt, map_location='cpu')
    image_model.load_state_dict(image_payload.get('model_state_dict', image_payload))
    image_model.eval()

    text_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(CONFIG['class_names']))
    text_payload = torch.load(text_ckpt, map_location='cpu')
    text_model.load_state_dict(text_payload.get('model_state_dict', text_payload))
    text_model.eval()

    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    image_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    traditional_metrics = evaluate_variant(
        traditional,
        test_samples,
        image_collection,
        text_collection,
        CONFIG,
        score_kwargs={
            'image_model': image_model,
            'text_model': text_model,
            'tokenizer': tokenizer,
            'transform': image_transform,
        },
    )
    results['variants']['traditional'] = traditional_metrics
else:
    print('Traditional checkpoints missing; skipping traditional baseline run.')

In [ ]:
best_alpha_idx = max(range(len(results['alpha_sweep']['macro_f1'])), key=lambda i: results['alpha_sweep']['macro_f1'][i])
results['best_alpha'] = results['alpha_sweep']['alphas'][best_alpha_idx]

save_results(results, str(RESULTS_PATH))

plot_scoring_comparison(results, str(FIGURES_DIR / 'scoring_comparison.png'))
plot_alpha_sensitivity(results['alpha_sweep'], str(FIGURES_DIR / 'alpha_sensitivity.png'))
plot_confusion_matrices(results, CONFIG['class_names'], str(FIGURES_DIR / 'confusion_matrices_phase2.png'))
phase1_macro_f1 = 0.8177
best_phase2 = max(v.get('macro_f1', 0.0) for k, v in results['variants'].items() if k != 'traditional')
plot_phase2_vs_phase1(best_phase2, phase1_macro_f1, str(FIGURES_DIR / 'phase2_vs_phase1.png'))

In [ ]:
print('=' * 60)
print('PHASE 2 RAC EXPERIMENT - RESULTS SUMMARY')
print('=' * 60)
print('Variant              | Accuracy | Macro F1 | TTR F1 | Latency')
print('---------------------|----------|----------|--------|--------')
for name, metrics in results['variants'].items():
    ttr_f1 = metrics.get('per_class_f1', {}).get('TTR', 0.0)
    print(f"{name:<20} | {metrics['accuracy']*100:6.2f}% | {metrics['macro_f1']:.4f} | {ttr_f1:.4f} | {metrics['latency_ms_per_sample']:.2f} ms")
print('=' * 60)
print(f"Best alpha (local DNDS): {results['best_alpha']}")